# Construction of pYPKµ

Simulate the two PCRs and homologous assembly that results in the **2249 bp circular pYPKµ plasmid**.

Use a Python kernel with **pydna**.



In [1]:
from pydna.amplify import pcr
from pydna.assembly2 import Assembly
from pydna.readers import read
from pydna.parsers import parse_primers

In [2]:
p = {}

p[1816], p[1815], p[1818], p[1817] = parse_primers('''

>1816_Ori_amp_fw 16-mer
tttccataggctccgc

>1815_Ori-amp_rv 20-mer
tatgtatccgctcatgagac

>1818_prom-crpS_fw 49-mer
TTGTCTCATGAGCGGATACATAataaaaaatttatttgctttcaggtac

>1817_prom-crpS_rv 40-mer
AGGGGGGCGGAGCCTATGGAAAttaacgagtgccgtaaac

''')

## 1. Verify the starting templates
PCR tmeplates are **pUCmu (1669 bp)** and **pYPKa (3128 bp)**. 

he literal checksums below were verified against the local GenBank files.

In [3]:
pucmu = read("pUCmu.gb")
assert pucmu.seguid() == "cdseguid=jEg2G-c5DZWICcghJ_xmafqcsTQ", "checksum does not match."
pucmu

Dseqrecord(o1669)

In [4]:
pypka = read("pYPKa.gb")
assert pypka.seguid() == "cdseguid=IFLMpKpCGZio0R0YkGSfqPaKKiw", "checksum does not match."
pypka

Dseqrecord(o3128)

## 2. PCR of the origin and ampicillin resistance region
Use forward primer **1816** and reverse primer **1815** on pUCmu. The recorded product is **1,536 bp**. Primers are stored in the local dictionary by their original numbers.

In [5]:
fp = p[1816]
rp = p[1815]
ori_amp = pcr(fp, rp, pucmu)
assert len(ori_amp) == 1536
assert ori_amp.seguid() == "ldseguid=XFYraLAaryuZlN1GqraV32i4_Uc", "PCR product differs from the recorded design."
ori_amp

Amplicon(1536)

## 3. PCR of the promoter–crpS region
Use primers **1818** and **1817** on pYPKa. The primer tails introduce overlaps with the first PCR product; the recorded product is **757 bp**.

The source note's forward/reverse labels are inconsistent between its table and drawing. This cell uses the table's order (1818, 1817); the double-stranded checksum verifies the product independently of orientation.

In [6]:
fp = p[1818]
rp = p[1817]
promoter_crps = pcr(fp, rp, pypka)
assert len(promoter_crps) == 757
assert promoter_crps.seguid() == "ldseguid=VcsSio0iH__oenir_UK_-M4rDsY", "PCR product differs from the recorded design."
promoter_crps

Amplicon(757)

## 4. Assemble pYPKµ
Use the source assembly note's minimum overlap of **13 bp**. Require exactly one circular candidate. The resulting assembly uses **22 bp at each junction**, giving 1,536 + 757 − 22 − 22 = **2,249 bp**.

In [7]:
asm = Assembly((ori_amp, promoter_crps), limit=13)
candidate, *rest = asm.assemble_circular()
assert not rest, "More than one circular assembly candidate."
assert candidate.circular
assert len(candidate) == 2249
candidate.name = "pYPKµ"
candidate.figure()

 -|1536bp_PCR_prod|22
|                  \/
|                  /\
|                  22|757bp_PCR_prod|22
|                                    \/
|                                    /\
|                                    22-
|                                       |
 ---------------------------------------

## 5. Compare with the independently saved plasmid
The expected checksum comes from both the assembly note and `pYPKµ 2249bp.gb`. The assembly note calls its FASTA record `pUKµ`; the notebook retains the project and GenBank name **pYPKµ**.

No rotation or synchronization is applied. Matching circular checksums establish sequence identity despite a different start position or orientation, without asserting annotation identity.

In [8]:
reference = read("pYPKµ.gb")
assert reference.seguid() == "cdseguid=-nmZwpMigpho3Bz361ay2keIGTc", "checksum does not match."
assert candidate.seguid() == "cdseguid=-nmZwpMigpho3Bz361ay2keIGTc", "Assembly differs from the recorded pYPKµ."
assert candidate.seguid() == reference.seguid()

## Check your understanding
Why is the assembled plasmid 44 bp shorter than the sum of the PCR products? Complete the calculation below.

In [9]:
overlap_per_junction = 22
expected_length = len(ori_amp) + len(promoter_crps) - 2 * overlap_per_junction
assert expected_length == len(candidate)

The shared overlaps are counted once in the assembled circle.

**Common pitfall:** a missing input file means the notebook kernel is running in a different folder. Set its working directory to this project before running the cells.

**Optional extension:** inspect `ori_amp.figure()` and `promoter_crps.figure()` to distinguish primer tails from template-binding regions.

## Local source records
- `PCR 1816 1815 pUCmu = ori amp.md`
- `PCR 1818 1817 pYPKa = promoter-crpS.md`
- `assembly pYPKµ = crp ori amp.md`
- `pUCmu 1669bp.gb`, `pYPKa 3128bp.gb`, and `pYPKµ 2249bp.gb`
